In [5]:
# Kaggle API 토큰은 이 셀에 적지 않는다. 아래 getpass 셀에서 입력받아 환경변수로만 사용한다.
# (토큰이 노트북/저장소에 남지 않게 하기 위해 평문 토큰 줄을 제거함)
import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

In [3]:
import os

In [4]:
import getpass, kagglehub
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Kaggle API token: ')    # 입력창에 직접 붙여넣기 (노트북에 값이 남지 않음)
kagglehub.competition_download('global-wheat-detection', output_dir=D)
print(os.listdir(D))

Kaggle API token:  ········


NameError: name 'D' is not defined

In [6]:
import shutil
shutil.copytree(path, './dataset', dirs_exist_ok=True)

NameError: name 'path' is not defined

In [3]:
import torch
print(torch.cuda.is_available())

True


In [28]:
!pip install --no-cache-dir torch torchvision --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [1]:
# torchvision은 설치된 torch(2.7.1+cu118)와 짝이 맞는 cu118 빌드여야 함
!pip install torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu118
!pip install pandas pillow matplotlib scikit-learn kagglehub ensemble-boxes

Looking in indexes: https://download.pytorch.org/whl/cu118
  Obtaining dependency information for torchvision==0.22.1 from https://download-r2.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://files.pythonhosted.org/packages/3b/2d/ede717bc1144f63886c21fd349bb95860b0d1a21149ff16f2bb362b612b6/pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 11.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 6.6 MB/s eta 0:00:0000:0100:01m
  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/63/be/dfb6cc9329d0bbe76dadda8a1113d026bb996ec76c509221368dc68349f4/pandas-3.0.6-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
  Obtaining

In [1]:
RUN pip install --no-cache-dir numpy torch torchvision --index-url https://download.pytorch.org/whl/cu118
RUN pip install --no-cache-dir pandas pillow matplotlib scikit-learn kagglehub ensemble-boxes seaborn

SyntaxError: invalid syntax (2910657628.py, line 1)

In [6]:
import os, glob, time, numpy as np, pandas as pd, torch
from PIL import Image
from torchvision.transforms import v2 as T
from torchvision import tv_tensors

device = torch.device('cuda')
print(torch.__version__, torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info(); print(f'VRAM 여유 {free/1e9:.1f} / {total/1e9:.1f} GB')   # Windows 화면 출력이 일부 점유함
W = H = 1024
D = './dataset'            # 노트북이 Kaggle/P1에 있을 때

ModuleNotFoundError: No module named 'pandas'

In [5]:
import os, shutil
src = '/home/jovyan/.cache/kagglehub/competitions/global-wheat-detection'
if not os.path.exists('./dataset/train.csv'):
    shutil.copytree(src, './dataset', dirs_exist_ok=True)
print(os.listdir('./dataset'))        # train, test, train.csv, sample_submission.csv 등이 있어야 함

['sample_submission.csv', 'test', 'train', 'train.csv']


In [10]:
import numpy as np, pandas as pd

IMG_W = IMG_H = 1024
df = pd.read_csv('./dataset/train.csv')
print(df.shape, df['image_id'].nunique(), df['source'].value_counts().to_dict())

xywh = np.stack(df['bbox'].apply(lambda s: np.fromstring(s.strip('[]'), sep=',')))
df[['x', 'y', 'w', 'h']] = xywh

# 이상한 박스 제거
df['area'] = df['w'] * df['h']
df['ar'] = df['w'] / df['h']
keep = (
    (df['w'] >= 5) & (df['h'] >= 5)               # 최소 변 5px
    & (df['area'] <= 100000)                      # 이삭 하나로 보기 어려운 큰 박스. 1024x1024 이미지의 약 10%
    & (df['ar'].between(0.2, 5))                  # 가로세로비 극단값 제외
)
print('제거:', (~keep).sum(), '/', len(df))
df = df[keep].copy()

# 이미지 경계 안으로 클리핑
df['x2'] = (df['x'] + df['w']).clip(upper=IMG_W)
df['y2'] = (df['y'] + df['h']).clip(upper=IMG_H)
df['x'] = df['x'].clip(lower=0); df['y'] = df['y'].clip(lower=0)
df['w'] = df['x2'] - df['x']; df['h'] = df['y2'] - df['y']

# 정규화된 중심좌표 (저장용)
df['cx'] = (df['x'] + df['w'] / 2) / IMG_W
df['cy'] = (df['y'] + df['h'] / 2) / IMG_H
df['nw'] = df['w'] / IMG_W
df['nh'] = df['h'] / IMG_H

# 학습/검증 분할: 반드시 박스가 아니라 이미지 단위로
ids = df['image_id'].unique()
rng = np.random.RandomState(41)                    # 가이드: 아무 정수
rng.shuffle(ids)
n_val = int(len(ids) * 0.15)                         # 가이드: 0.1~0.2
val_ids, train_ids = ids[:n_val], ids[n_val:]

ModuleNotFoundError: No module named 'pandas'

In [10]:
import torch
from PIL import Image
from torchvision.transforms import v2 as T
from torchvision import tv_tensors

class WheatDataset(torch.utils.data.Dataset):
    def __init__(self, df, image_ids, img_dir, train=True):
        self.groups = {k: g for k, g in df[df['image_id'].isin(image_ids)].groupby('image_id')}
        self.ids = list(self.groups)
        self.img_dir, self.train = img_dir, train
        self.tf = T.Compose([
            T.RandomHorizontalFlip(0.5),            # 가이드: 0.5
            T.RandomVerticalFlip(0.5),              # 가이드: 0.5
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),   # 가이드: 0.1~0.3
        ]) if train else None

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        iid = self.ids[i]; g = self.groups[iid]
        img = tv_tensors.Image(Image.open(f'{self.img_dir}/{iid}.jpg').convert('RGB'))
        # 정규화 중심좌표 -> 절대 픽셀 xyxy (R-CNN 입력 형식)
        cx, cy, w, h = (g[c].values for c in ['cx', 'cy', 'nw', 'nh'])
        xyxy = np.stack([(cx - w/2) * IMG_W, (cy - h/2) * IMG_H,
                         (cx + w/2) * IMG_W, (cy + h/2) * IMG_H], 1)
        boxes = tv_tensors.BoundingBoxes(xyxy, format='XYXY', canvas_size=(IMG_H, IMG_W), dtype=torch.float32)
        target = {'boxes': boxes, 'labels': torch.ones(len(boxes), dtype=torch.int64)}   # 클래스 1 = wheat
        if self.tf: img, target = self.tf(img, target)
        return img.as_subclass(torch.Tensor).float() / 255.0, {'boxes': target['boxes'].as_subclass(torch.Tensor), 'labels': target['labels']}

collate = lambda b: tuple(zip(*b))

In [11]:
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = fasterrcnn_mobilenet_v3_large_fpn(
    weights='DEFAULT',
    min_size=32, max_size=512,              # 가이드: 512 (CPU). 1024는 정확하지만 매우 느림. 이미지/박스는 모델이 알아서 리사이즈
    trainable_backbone_layers=1,           # 가이드: 0~3. 작을수록 빠르고 CPU에 유리
    box_detections_per_img=300,              # 가이드: 이미지당 이삭이 많으니 100~300
)
in_f = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_f, 2)    # 배경 + wheat
model.to(device)    # GPU로 이동 (옵티마이저 생성 전에 실행)


Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth" to /home/jovyan/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth


100%|██████████| 74.2M/74.2M [00:02<00:00, 33.5MB/s]


In [12]:
torch.set_num_threads(2)                          # 가이드: CPU 코어 수 근처

train_ds = WheatDataset(df, train_ids[:200], './dataset/train', train=True)   # 가이드: 처음엔 200~300장으로 시간 측정
loader = torch.utils.data.DataLoader(train_ds, batch_size=3, shuffle=True,   # 가이드: 2~4 (CPU 메모리)
                                     collate_fn=collate, num_workers=1, pin_memory=True)      # 가이드: 0~2

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)          # 가이드: lr 0.005, wd 5e-4

import time
for epoch in range(1):                            # 가이드: 처음엔 1로 시간 확인
    model.train(); t0 = time.time(); total = 0
    for imgs, tgts in loader:
        imgs = [i.to(device, non_blocking=True) for i in imgs]
        tgts = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in tgts]
        loss = sum(model(imgs, tgts).values())
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f'epoch {epoch}  loss {total/len(loader):.4f}  {time.time()-t0:.0f}s')

ERROR: Unexpected bus error encountered in worker. This might be caused by insufficient shared memory (shm).
 

RuntimeError: DataLoader worker (pid 254) is killed by signal: Bus error. It is possible that dataloader's workers are out of shared memory. Please try to raise your shared memory limit.

In [21]:
SCORE_THR = 0.5                                     # 가이드: 0.3~0.6 (낮을수록 박스가 많음)

@torch.no_grad()
def predict(img_dir, image_ids):
    model.eval(); rows = []
    for iid in image_ids:
        img = T.functional.to_image(Image.open(f'{img_dir}/{iid}.jpg').convert('RGB')).float() / 255.0
        out = model([img.to(device)])[0]
        s = out['scores'].cpu().numpy(); b = out['boxes'].cpu().numpy()
        m = s >= SCORE_THR
        parts = [f'{sc:.4f} {x1:.1f} {y1:.1f} {x2-x1:.1f} {y2-y1:.1f}'      # 제출은 x y w h
                 for sc, (x1, y1, x2, y2) in zip(s[m], b[m])]
        rows.append({'image_id': iid, 'PredictionString': ' '.join(parts)})
    return pd.DataFrame(rows)

test_ids = [f[:-4] for f in sorted(os.listdir('./dataset/test'))]
sub = predict('./dataset/test', test_ids)
sub.to_csv('submission.csv', index=False)